# Classification NBA Model

## Configuration

## Imports

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
)
from nba_ou.data_preparation.missing_data.clean_df_for_training import (
    clean_dataframe_for_training
)
from sklearn.model_selection import KFold, cross_validate, train_test_split
from xgboost import XGBClassifier


In [2]:
from nba_ou.modeling.modeling import (
    split_latest_dates_holdout,
    make_walk_forward_last_n_seasons_splits,
    validate_time_splits,
    make_test_anchored_walk_forward_splits,
    assert_valid_time_splits,
    save_model_bundle,
    load_model_bundle
)

In [3]:
nan_threshold = 50.0
max_na_per_row = 5

## Load Data

In [4]:
data_path = "/home/adrian_alvarez/Projects/NBA_over_under_predictor/data/train_data/"
name = "all_odds_training_data_until_20260316.csv"

path = data_path + name

df_stats = pd.read_csv(path)

dtype_dict = {col: str for col in df_stats.columns if "ID" in col.upper()}

df_stats = pd.read_csv(
    path,
    dtype=dtype_dict
)
df_stats['GAME_DATE'] = pd.to_datetime(df_stats['GAME_DATE']).dt.strftime('%Y-%m-%d')

In [5]:
import time
time.sleep(4.7*3600)

In [6]:
df_to_train = clean_dataframe_for_training(df_stats, nan_threshold=nan_threshold, max_na_per_row=max_na_per_row, create_missing_flags=False, verbose=1, keep_columns=['GAME_DATE'])

STARTING DATAFRAME CLEANING PIPELINE
Starting basic cleaning with 11241 rows
Basic cleaning complete: 8632 rows remaining

Starting advanced column cleaning with 3240 columns

Advanced column cleaning complete: 3240 → 2137 columns (1103 removed)


Dropping NA rows for SEASON_YEAR 2017...
   Removed 0 rows with NaN values from 2017 season

Applying missing data policy...

Missing Data Policy Report:
  Rows dropped: 0 (0.0%)
  Critical columns requiring data: 4
  Columns zero-filled: 112
  Infer pairs applied: 20/136
  Remaining NaN cells: 944087

Dropping rows with more than 5 NaN values...
Removed 3945 rows exceeding NaN threshold
CLEANING COMPLETE
Final shape: (4687, 2137)


In [7]:
# Count NAs per column
na_counts = df_to_train.isna().sum()

# Get most common SEASON_YEAR for nulls in each column
most_common_season = []
for col in df_to_train.columns:
    if na_counts[col] > 0:
        # Get rows where this column is null
        null_rows = df_to_train[df_to_train[col].isna()]
        if len(null_rows) > 0 and 'SEASON_YEAR' in df_to_train.columns:
            # Find most common SEASON_YEAR for these null rows
            common_season = null_rows['SEASON_YEAR'].mode()
            most_common_season.append(common_season.iloc[0] if len(common_season) > 0 else None)
        else:
            most_common_season.append(None)
    else:
        most_common_season.append(None)

na_counts_df = pd.DataFrame({
    'Column': na_counts.index,
    'NA_Count': na_counts.values,
    'NA_Percentage': (na_counts.values / len(df_to_train) * 100).round(2),
    'Most_Common_Season_Year': most_common_season
}).sort_values('NA_Count', ascending=False)

# Show only columns with NAs
na_counts_df[na_counts_df['NA_Count'] > 0]

,Column,NA_Count,NA_Percentage,Most_Common_Season_Year
1946,total_consensus_pct_under,67,1.43,2024.0
1947,spread_consensus_pct_home,63,1.34,2024.0
1953,ml_consensus_opener_price_away,14,0.30,2025.0
1954,ml_consensus_opener_price_home,14,0.30,2025.0
1801,total_consensus_pct_over_TREND_SLOPE_LAST_5_HO...,9,0.19,2023.0
1807,spread_consensus_pct_home_TREND_SLOPE_LAST_5_H...,9,0.19,2023.0
1805,spread_consensus_pct_away_TREND_SLOPE_LAST_5_H...,9,0.19,2023.0
1803,total_consensus_pct_under_TREND_SLOPE_LAST_5_H...,8,0.17,2023.0
1223,spread_consensus_pct_away_TREND_SLOPE_LAST_5_H...,5,0.11,2023.0
556,total_consensus_pct_over_TREND_SLOPE_LAST_5_HO...,5,0.11,2023.0


In [8]:
BET365_LINE_COL =  "TOTAL_LINE_bet365"
# BET365_LINE_COL =  "total_bet365_line_over"

# Ensure scoring line and target exist (avoid NaN-driven undefined betting accuracy).
df_to_train = df_to_train.dropna(subset=[BET365_LINE_COL, "TOTAL_POINTS"]).copy()

In [9]:
df_to_train['GAME_DATE'] = pd.to_datetime(df_to_train['GAME_DATE'])
df_to_train = df_to_train.sort_values("GAME_DATE").reset_index(drop=True)

In [10]:
#count games per season
games_per_season = df_to_train.groupby('SEASON_YEAR').size()
print(games_per_season)

SEASON_YEAR
2021    1158
2022    1173
2023     170
2024    1248
2025     938
dtype: int64


## Train / Test

In [11]:
from sklearn.model_selection import KFold, cross_validate, cross_val_predict, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, make_scorer, root_mean_squared_error
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from sklearn.base import clone

In [12]:
df_dev, df_test_final = split_latest_dates_holdout(
    df=df_to_train,
    date_col="GAME_DATE",
    test_size=0.05,
)

print(f"Development set size: {len(df_dev)}")
print(f"Final test set size: {len(df_test_final)}")
print("Final test date range:",
      df_test_final["GAME_DATE"].min(), "->", df_test_final["GAME_DATE"].max())

Development set size: 4452
Final test set size: 235
Final test date range: 2026-02-08 00:00:00 -> 2026-03-16 00:00:00


In [13]:
TARGET_COL = "TOTAL_POINTS"

EXCLUDE_COLS = [
    "TOTAL_POINTS",
    "SEASON_YEAR",
    "GAME_DATE",
]

X_dev = df_dev.drop(columns=EXCLUDE_COLS, errors="ignore")
y_dev = pd.to_numeric(df_dev[TARGET_COL], errors="coerce")

X_test_final = df_test_final.drop(columns=EXCLUDE_COLS, errors="ignore")
y_test_final = pd.to_numeric(df_test_final[TARGET_COL], errors="coerce")

print(f"X_dev shape: {X_dev.shape}")
print(f"X_test_final shape: {X_test_final.shape}")

X_dev shape: (4452, 2134)
X_test_final shape: (235, 2134)


In [14]:
from nba_ou.modeling.scorers import OverUnderScorerTotalPoints, over_under_betting_accuracy_total_points, evaluate_total_points_thresholds
    
ou_scorer = OverUnderScorerTotalPoints(BET365_LINE_COL)

scoring = {
    "MSE": "neg_mean_squared_error",
    "RMSE": "neg_root_mean_squared_error",
    "MAE": "neg_mean_absolute_error",
    "R2": "r2",
    "OU_Betting_Accuracy": ou_scorer,
}

def print_metrics(cv_results):
    for sc in scoring.keys():
        train_key = f"train_{sc}"
        test_key = f"test_{sc}"

        train_val = cv_results[train_key].mean()
        test_val = cv_results[test_key].mean()

        if sc in {"MSE", "RMSE", "MAE"}:
            train_val = -train_val
            test_val = -test_val

        if sc == "OU_Betting_Accuracy":
            print(f"Train {sc}: {train_val:.2%}")
            print(f"Validation {sc}: {test_val:.2%}")
        else:
            print(f"Train {sc}: {train_val:.5f}")
            print(f"Validation {sc}: {test_val:.5f}")
        print()

In [15]:
splits, fold_info = make_test_anchored_walk_forward_splits(
    df=df_dev,
    date_col="GAME_DATE",
    season_col="SEASON_YEAR",
    test_games=30,
    step_games_between_tests=75,
    train_games=2500,
    min_train_games=2000,
    max_folds=14,
    verbose=1,
)


Created 14 test-anchored walk-forward folds
 fold  train_n_games  test_n_games train_start_date train_end_date test_start_date test_end_date  test_season
    1           2500            31       2021-12-21     2024-12-21      2024-12-22    2024-12-26         2024
    2           2500            33       2022-01-07     2025-01-06      2025-01-07    2025-01-11         2024
    3           2500            39       2022-01-23     2025-01-22      2025-01-23    2025-01-27         2024
    4           2500            31       2022-02-08     2025-02-06      2025-02-07    2025-02-10         2024
    5           2500            30       2022-03-02     2025-02-26      2025-02-27    2025-03-02         2024
    6           2500            31       2022-03-18     2025-03-12      2025-03-13    2025-03-16         2024
    7           2500            30       2022-04-01     2025-03-26      2025-03-27    2025-03-30         2024
    8           2500            35       2022-04-27     2025-04-09      2025

In [16]:
season_bl = DummyRegressor(strategy="mean")

cv_results = cross_validate(
    season_bl,
    X_dev,
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1,
)

print("DummyRegressor baseline")
print_metrics(cv_results)

DummyRegressor baseline
Train MSE: 379.45524
Validation MSE: 398.81929

Train RMSE: 19.47943
Validation RMSE: 19.81787

Train MAE: 15.53717
Validation MAE: 15.95272

Train R2: 0.00000
Validation R2: -0.05597

Train OU_Betting_Accuracy: 49.98%
Validation OU_Betting_Accuracy: 46.68%



In [17]:
lr = LinearRegression()

cv_results = cross_validate(
    lr,
    X_dev.fillna(0),   # LR cannot handle NaNs
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1,
)

print("Linear Regression")
print_metrics(cv_results)

Linear Regression
Train MSE: 119.77496
Validation MSE: 37824096.71137

Train RMSE: 10.94294
Validation RMSE: 1767.56726

Train MAE: 8.50358
Validation MAE: 1068.53637

Train R2: 0.68431
Validation R2: -173310.04489

Train OU_Betting_Accuracy: 77.97%
Validation OU_Betting_Accuracy: 49.25%



In [18]:
xgb_reg = XGBRegressor(
    max_depth=4,
    learning_rate=0.057,
    n_estimators=750,
    subsample=0.8,
    colsample_bytree=0.86,
    reg_alpha=0.57,
    reg_lambda=1.78,
    min_child_weight=5.48,
    gamma=1.77,
    n_jobs=-2,          # choose your CPU count
    random_state=16,
)

cv_results = cross_validate(
    xgb_reg,
    X_dev,
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1,          # leave at 1 because XGB already parallelizes internally
)

print("XGBoost")
print_metrics(cv_results)

XGBoost
Train MSE: 2.90466
Validation MSE: 336.66726

Train RMSE: 1.70368
Validation RMSE: 18.21191

Train MAE: 1.30134
Validation MAE: 14.67653

Train R2: 0.99234
Validation R2: 0.08072

Train OU_Betting_Accuracy: 98.68%
Validation OU_Betting_Accuracy: 46.98%



In [19]:
xgb_reg.fit(X_dev, y_dev)

y_pred_test_total = xgb_reg.predict(X_test_final)

mse = mean_squared_error(y_test_final, y_pred_test_total)
rmse = root_mean_squared_error(y_test_final, y_pred_test_total)
mae = mean_absolute_error(y_test_final, y_pred_test_total)

betting_line = X_test_final[BET365_LINE_COL].to_numpy(dtype=float)

ou_acc = over_under_betting_accuracy_total_points(
    y_test_final,
    y_pred_test_total,
    betting_line,
)

print("Final test metrics")
print(f"MSE: {mse:.5f}")
print(f"RMSE: {rmse:.5f}")
print(f"MAE: {mae:.5f}")
print(f"OU_Betting_Accuracy: {ou_acc:.2%}")

Final test metrics
MSE: 327.68399
RMSE: 18.10204
MAE: 14.33107
OU_Betting_Accuracy: 46.29%


In [20]:
results_df, y_pred_test_total = evaluate_total_points_thresholds(
    model=xgb_reg,
    X_test=X_test_final,
    y_test_total=y_test_final,
    line_col=BET365_LINE_COL,
    thresholds=range(0, 11),
)

display(
    results_df.style.format(
        {"pct_of_test": "{:.1%}", "ou_betting_accuracy": "{:.2%}"}
    )
)

,threshold_abs_pred_edge_gt,n_games,pct_of_test,ou_betting_accuracy
0,0,235,100.0%,46.29%
1,1,190,80.9%,44.57%
2,2,138,58.7%,44.36%
3,3,104,44.3%,45.00%
4,4,81,34.5%,40.26%
5,5,63,26.8%,40.00%
6,6,43,18.3%,45.24%
7,7,28,11.9%,48.15%
8,8,19,8.1%,57.89%
9,9,13,5.5%,61.54%


# OPTUNA

In [21]:
from nba_ou.modeling.optuna_total_points import (
    tune_xgb_total_points_optuna,
    summarize_optuna_trials,
    fit_best_xgb_total_points,
)

study = tune_xgb_total_points_optuna(
    X=X_dev,
    y=y_dev,
    splits=splits,
    line_col=BET365_LINE_COL,
    n_trials=80,
    timeout=4.75 * 3600,
    objective_name="reg:squarederror",   # second run: reg:pseudohubererror
    study_name="xgb_total_points_mae",
)

print("Best CV MAE:", study.best_value)
print("Best params:")
for k, v in study.best_params.items():
    print(f"{k}: {v}")

print("\nSecondary metrics from best trial:")
print("Mean RMSE:", study.best_trial.user_attrs.get("mean_rmse"))
print("Mean R2:", study.best_trial.user_attrs.get("mean_r2"))
print("Mean OU accuracy:", study.best_trial.user_attrs.get("mean_ou_acc"))
print("Mean best_iteration:", study.best_trial.user_attrs.get("mean_best_iteration"))

trials_df = summarize_optuna_trials(study)
display(
    trials_df.head(15).style.format(
        {
            "value_mae": "{:.4f}",
            "mean_rmse": "{:.4f}",
            "mean_r2": "{:.4f}",
            "mean_ou_acc": "{:.2%}",
        }
    )
)

[I 2026-03-18 04:22:15,265] A new study created in memory with name: xgb_total_points_mae


  0%|          | 0/80 [00:00<?, ?it/s]

[I 2026-03-18 04:29:54,079] Trial 0 finished with value: 13.603661694022204 and parameters: {'max_depth': 2, 'min_child_weight': 14.839989016734002, 'gamma': 1.6521043697435434, 'subsample': 0.5682407800531226, 'colsample_bytree': 0.6123279759064977, 'learning_rate': 0.015902381379451283, 'reg_alpha': 1.8771791376898666, 'reg_lambda': 0.2766344129879233}. Best is trial 0 with value: 13.603661694022204.
[I 2026-03-18 04:40:13,695] Trial 1 finished with value: 13.518354812043368 and parameters: {'max_depth': 2, 'min_child_weight': 35.38241645406617, 'gamma': 1.6910441407044758, 'subsample': 0.5811969357559948, 'colsample_bytree': 0.7751882300061779, 'learning_rate': 0.013902617405829187, 'reg_alpha': 0.06701717253228541, 'reg_lambda': 0.6196026989845951}. Best is trial 1 with value: 13.518354812043368.
[I 2026-03-18 04:45:45,688] Trial 2 finished with value: 13.50024836198298 and parameters: {'max_depth': 4, 'min_child_weight': 13.12932060704323, 'gamma': 0.6451864311430185, 'subsample':

,trial,value_mae,mean_rmse,mean_r2,mean_ou_acc,mean_best_iteration,max_depth,min_child_weight,gamma,subsample,colsample_bytree,learning_rate,reg_alpha,reg_lambda
0,31,13.2787,17.0052,0.2059,59.01%,87,3,5.972148,1.418688,0.659190,0.830032,0.068509,1.510938,10.681136
1,33,13.3418,17.0853,0.1988,57.08%,69,3,7.509241,1.569755,0.657508,0.883836,0.069562,1.421353,4.487127
2,46,13.3521,17.0199,0.2041,57.67%,96,3,7.702166,1.101562,0.673162,0.730383,0.059403,3.224735,1.742886
3,41,13.3783,17.0310,0.2013,57.51%,63,3,9.011110,1.418935,0.714730,0.880981,0.079691,1.736071,5.095178
4,34,13.3790,17.0872,0.1987,54.28%,71,3,7.644264,1.520543,0.657706,0.883654,0.069727,1.337863,5.280920
5,37,13.3945,17.0812,0.1984,58.03%,102,2,7.113692,1.381937,0.646124,0.899773,0.054899,1.235861,2.924629
6,12,13.4023,17.1297,0.1963,55.76%,120,3,5.135323,2.917899,0.917147,0.881900,0.077739,0.177752,25.311357
7,22,13.4026,17.0962,0.1973,55.92%,99,3,5.159339,2.991644,0.915718,0.842200,0.067208,0.035584,13.890055
8,42,13.4107,17.1919,0.1891,57.43%,49,4,9.118040,1.425250,0.723781,0.866530,0.069954,1.414992,5.060190
9,23,13.4134,17.0635,0.2007,53.79%,99,3,5.235557,2.981378,0.925093,0.845851,0.060674,0.039052,12.271712


In [22]:
best_model = fit_best_xgb_total_points(
    X_dev=X_dev,
    y_dev=y_dev,
    study=study,
    objective_name="reg:squarederror",
)

y_pred_test_total = best_model.predict(X_test_final)

mse = mean_squared_error(y_test_final, y_pred_test_total)
rmse = root_mean_squared_error(y_test_final, y_pred_test_total)
mae = mean_absolute_error(y_test_final, y_pred_test_total)

betting_line = X_test_final[BET365_LINE_COL].to_numpy(dtype=float)

ou_acc = over_under_betting_accuracy_total_points(
    y_true=y_test_final,
    y_pred=y_pred_test_total,
    betting_line=betting_line,
)

print("Final test metrics")
print(f"MSE: {mse:.5f}")
print(f"RMSE: {rmse:.5f}")
print(f"MAE: {mae:.5f}")
print(f"OU_Betting_Accuracy: {ou_acc:.2%}")

Final test metrics
MSE: 304.86661
RMSE: 17.46043
MAE: 13.79318
OU_Betting_Accuracy: 50.22%


In [23]:
# Limit to Season year 2023 onwards
train_data = df_to_train[df_to_train["SEASON_YEAR"] >= 2023].copy()

In [24]:
from nba_ou.modeling.modeling import ModelBundleMetadata, ModelInfo, TrainingMetrics

X_full = train_data.drop(columns=EXCLUDE_COLS, errors="ignore")
y_full = pd.to_numeric(train_data[TARGET_COL], errors="coerce")


production_model = fit_best_xgb_total_points(
    X_dev=X_full,
    y_dev=y_full,
    study=study,
    objective_name="reg:squarederror",
)

latest_training_date = pd.to_datetime(train_data["GAME_DATE"]).max()
model_version = latest_training_date.strftime("%d_%m_%y")
model_name = f"three_seasons_xgb_total_points_{model_version}"

metadata = ModelBundleMetadata(
    model_info=ModelInfo(
        name=model_name,
        model_version=model_version,
        model_type="three_seasons_total_points",
        prediction_source="three_seasons_xgb_total_points",
        training_code_tag="1.0",
    ),
    training_metrics=TrainingMetrics(
        best_params=study.best_trial.params,
        mean_best_iteration=study.best_trial.user_attrs.get("mean_best_iteration"),
        cv_mae=float(study.best_value),
        cv_rmse=study.best_trial.user_attrs.get("mean_rmse"),
        cv_ou_acc=study.best_trial.user_attrs.get("mean_ou_acc"),
        final_test_mae=float(mae),
        final_test_rmse=float(rmse),
        final_test_ou_acc=float(ou_acc),
        nan_threshold=nan_threshold,
        max_na_per_row=max_na_per_row,
        train_date_min=train_data["GAME_DATE"].min().to_pydatetime(),
        train_date_max=train_data["GAME_DATE"].max().to_pydatetime(),
    ),
)

model_path, meta_path = save_model_bundle(
    model=production_model,
    feature_names=list(X_full.columns),
    out_dir="/home/adrian_alvarez/Projects/NBA_over_under_predictor/models/total_points/3_seasons/",
    metadata=metadata,
)

print(f"Production model trained on {len(X_full)} rows using fixed n_estimators from mean_best_iteration.")
print("Saved model :", model_path)
print("Saved metadata:", meta_path)


Production model trained on 2356 rows using fixed n_estimators from mean_best_iteration.
Saved model : /home/adrian_alvarez/Projects/NBA_over_under_predictor/models/total_points/3_seasons/three_seasons_xgb_total_points_16_03_26.json
Saved metadata: /home/adrian_alvarez/Projects/NBA_over_under_predictor/models/total_points/3_seasons/three_seasons_xgb_total_points_16_03_26.meta.json
